In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import importlib
from glob import glob
import scipy as sp
from time import time
import seaborn as sns
from matplotlib import pyplot as plt
import networkx as nx
import polars as pl
import pickle
import glob
from joblib import Parallel, delayed
import oracledb
import scienceplots
plt.style.use('science')

# Maps
import folium
import mapclassify
import geopandas as gp
import shapely
import shapely.affinity
from shapely.geometry import Point
import matplotlib


### Clusters

In [ ]:
sl_list = ['wildtype', 'Alpha', 'Eta', 'Delta', 'omicron']
lineage_sizes = np.array((27651, 70471, 671,  167630, 16172))
lineage_sizes_dict = dict(zip(*(sl_list, lineage_sizes)))

In [ ]:
n_random_clusters = 100
n_connected_nodes = np.zeros((len(sl_list), n_random_clusters))
cluster_sizes_dict_variant = {}
cluster_regions_dict_variant = {}
cluster_kommuner_dict_variant = {}
for (e, sl) in enumerate(sl_list): 
    cluster_sizes_dict = {}
    cluster_regions_dict = {}
    cluster_kommuner_dict = {}
    for i in tqdm(range(n_random_clusters)):
        nodelist_test_random = pl.scan_csv('../../clusters/' + sl + '/random_clusters/nodelist_cluster_' + str(i) + '.csv')
        n_connected_nodes[e, i] = len(nodelist_test_random.collect())
        cluster_sizes_dict[i] = nodelist_test_random.group_by(pl.col(
            'cluster_number')).len().select('len').collect().to_numpy()
        
        
        # Dicts for regions
        region_counts = len(nodelist_test_random.group_by(pl.col('cluster_number',
                                       'region')).len().group_by('cluster_number').len().filter(pl.col('len')
                                                                                                   >1).collect())
        cluster_regions_dict[i] = region_counts / len(cluster_sizes_dict[i])
        
    
        mean_kommune_counts = nodelist_test_random.group_by(pl.col('cluster_number',
                                       'kommune')).len().group_by(
                                                    'cluster_number'
                                                                ).len().collect().select('len').mean().item()
        cluster_kommuner_dict[i] = mean_kommune_counts
        
    cluster_sizes_dict_variant[sl] = cluster_sizes_dict
    cluster_regions_dict_variant[sl] = cluster_regions_dict
    cluster_kommuner_dict_variant[sl] = cluster_kommuner_dict

In [ ]:
# def get_random_cluster_statistics(rc_dict):
def filter_too_small(n):
    if n < 5:
        return np.nan
    else:
        return n
        
max_cluster_sizes = np.zeros((len(sl_list), n_random_clusters))
for (e, sl) in enumerate(sl_list): 
    for i in tqdm(range(n_random_clusters)):
        max_cluster_sizes[e, i] = np.max(cluster_sizes_dict_variant[sl][i].flatten())

In [ ]:
cmp = sns.color_palette()
cluster_sizes_sum_dict = {}
cluster_sizes_real_dict = {}
cluster_sizes_len_dict = {}
cluster_sizes_len_dict_real = {}
cluster_kommuner_dict_real = {}
cluster_regions_dict_real = {}
cluster_sizes_max_dict = {}
for sl in sl_list:
    nodelist_test = pl.scan_csv('../../clusters/' + sl + '/node_attributes/nodelist_clusters_macro_school.csv')
    cluster_sizes_real = nodelist_test.group_by(pl.col(
                                    'cluster_number')).len().select('len').collect()
    cluster_sizes_real = cluster_sizes_real.to_numpy().flatten()
    cluster_sizes_real_dict[sl] = cluster_sizes_real
    cluster_sizes_len_dict_real[sl] = nodelist_test.select('cluster_number').max().collect().item()
    
    cluster_sizes_dict = cluster_sizes_dict_variant[sl]
    sum_cluster_sizes = [np.sum(cs) for cs in cluster_sizes_dict.values()]
    cluster_sizes_sum_dict[sl] = sum_cluster_sizes  / lineage_sizes_dict[sl]
    cluster_sizes_len_dict[sl] = [len(cs) for cs in cluster_sizes_dict.values()]
    max_cluster_sizes = [np.max(cs) for cs in cluster_sizes_dict.values()]
    cluster_regions_dict_real[sl] = len(nodelist_test.group_by(pl.col('cluster_number',
                                       'region')).len().group_by('cluster_number').len().filter(pl.col('len')
                                                                                                   >1).collect())
    cluster_kommuner_dict_real[sl] = nodelist_test.group_by(pl.col('cluster_number',
                                       'kommune')).len().group_by(
                                                'cluster_number'
                                                            ).len().collect().select('len').mean().item()
    cluster_sizes_max_dict[sl] = max_cluster_sizes


### Maps

In [ ]:
cluster_dict = {}
largest_cluster_dict = {}
largest_cluster_dfs = {}
largest_cluster_idxs = {}
for sl in tqdm(sl_list):
    with open('../../clusters/' +
                    sl + '/cluster_dicts/settings_cluster_dict_macro_school.pickle', 'rb') as handle:
        b = pickle.load(handle)
        cluster_dict[sl] = b
    largest_cluster_idx = np.argmax(np.array([d['component_size'] for d in b.values()]))
    largest_cluster_idxs[sl] = largest_cluster_idx
    largest_cluster_dict[sl] = b[largest_cluster_idx]
    
    cluster_kommuner = pl.read_csv('../../clusters/' +
                    sl + 
                    '/node_attributes/nodelist_clusters_macro_school.csv').filter((pl.col('cluster_number') ==
                                                                                   largest_cluster_idx) & 
                                                                     (~pl.col('kommune')
                                                                      .is_null()))#.groupby(pl.col('kommune')).count()
    largest_cluster_dfs[sl] = cluster_kommuner
    
    
    
cluster_greater_than_5 = np.zeros(len(sl_list))
cluster_greater_than_10 = np.zeros(len(sl_list))
cluster_greater_than_20 = np.zeros(len(sl_list))
for i, sl in enumerate(sl_list):
    cluster_greater_than_5[i] = np.sum([c['component_size'] for c in cluster_dict[sl].values()])
    cluster_greater_than_10[i] = np.sum([c['component_size'] for 
                                         c in cluster_dict[sl].values() if c['component_size'] > 10])
    cluster_greater_than_20[i] = np.sum([c['component_size'] for 
                                         c in cluster_dict[sl].values() if c['component_size'] > 20])
    

In [ ]:
regions = gp.read_file('../../../Geo_data/kommuner_small.geojson')

In [ ]:
# fig, axes = plt.subplots(figsize = (10, 6), ncols = 2, nrows = 1)

macro_school = True

combined_cluster_df = pl.concat([largest_cluster_dfs['wildtype'], largest_cluster_dfs['Alpha'] , 
          largest_cluster_dfs['Eta'], largest_cluster_dfs['Delta']]).group_by(pl.col('kommune')).len()
cluster_size = combined_cluster_df.select(pl.col('len').sum()).item()
regions_cluster = regions.merge(combined_cluster_df.to_pandas(),
                                left_on = 'KOMNAVN',
                                right_on = 'kommune', 
                                how = 'left')

regions_cluster = regions_cluster.loc[regions_cluster['KOMNAVN'] != 'Bornholm']
regions_cluster.loc[regions_cluster['len']<=5, 'len']= np.nan

copenhagen_point = Point(12.5655, 55.6759)
odense_point = Point(10.3883, 55.39594)
aarhus_point = Point(10.21076, 56.1572)
aalborg_point = Point(9.9187, 57.0480)
city_labels = ['Copenhagen', 'Odense', 'Aarhus', 'Aalborg']
city_points = [copenhagen_point, odense_point, aarhus_point, aalborg_point]

d = {'col1' : city_labels,
     'geometry' : city_points}

gdf = gp.GeoDataFrame(d, crs=regions_cluster.crs)

city_points_x = [point.x for point in city_points]
city_points_y = [point.y for point in city_points]

In [ ]:
from collections import Counter
kommune_crossover_dict = {}
for sl in tqdm(sl_list):
    with open('../../clusters/' +
                    sl + '/cluster_dicts/kommune_crossover_dict_macro_school.pickle', 'rb') as handle:
        a = pickle.load(handle)
        kommune_crossover_dict[sl] = a
kommune_crossover_dict
largest_cluster_idxs = np.zeros(len(sl_list)).astype(int)
for i, b in enumerate(cluster_dict.values()):
    largest_cluster_idxs[i] = int(np.argmax(np.array([d['component_size'] for d in b.values()])))


counter_dict = dict(Counter(kommune_crossover_dict[sl][largest_cluster_idxs[-1]]['crossover_kommuner']))
arrows_df = pl.DataFrame([(key1[0], key1[1], val) for (key1,  val) in counter_dict.items()], 
            schema = ['From', 'To', 'Number']).filter(pl.col('Number')> 5).to_pandas()
# arrows_df

In [ ]:
import warnings
warnings.filterwarnings("ignore")
def draw_curve(p1, p2):
     """Return curves lines between two points p1 and p2."""
     #np.cosh(x) is equivalent to 0.5*(np.exp(x) - np.exp(-x))
     a = (p2[1] - p1[1])/ (np.cosh(p2[0]) - np.cosh(p1[0]))
     b = p1[1] - a * np.cosh(p1[0])
     x = np.linspace(p1[0], p2[0], 100)
     y = a * np.cosh(x) + b
     return x, y
curve_dict_variant = {}
regions_cluster_dict_variant = {}
regions_cluster_outbreak_variant_dict = {}
regions_cluster_variant_dict = {}
for s, sl in enumerate(sl_list):
    curve_dict_variant[sl] = {}
    df = largest_cluster_dfs[sl].group_by(pl.col('kommune')).len()
    regions_cluster = regions.merge(df.to_pandas(),
                                    left_on = 'KOMNAVN',
                                    right_on = 'kommune', 
                                    how = 'left')
    cluster_size = df.select(pl.col('len').sum()).item()
    regions_cluster = regions_cluster.loc[regions_cluster['KOMNAVN'] != 'Bornholm']
    regions_cluster['len'] = regions_cluster['len'].apply(filter_too_small)
    
    regions_joined = regions_cluster[['KOMKODE', 'KOMNAVN', 'geometry']].dissolve(by = 'KOMKODE')
    regions_joined['centroid'] = regions_joined.geometry.centroid
    regions_cluster = regions_cluster.merge(regions_joined.filter(['KOMKODE', 'centroid']), on = 'KOMKODE', how = 'left')
    regions_cluster_outbreak = regions_cluster.loc[~pd.isna(regions_cluster['len'])]
    
    regions_cluster_outbreak['centroid'].loc[regions_cluster_outbreak['REGIONNAVN'] == 'Region Hovedstaden'] = Point(12.55238, 55.67532)
    
    
    variant_crossovers = set(kommune_crossover_dict[sl][largest_cluster_idxs[s]]['crossover_kommuner'])

    crossover_counts_dict = {}
    for i in kommune_crossover_dict[sl][largest_cluster_idxs[s]]['crossover_kommuner']: 
        crossover_counts_dict[i] = 0
    for i in kommune_crossover_dict[sl][largest_cluster_idxs[s]]['crossover_kommuner']: 
        crossover_counts_dict[i] += 1
        
    centroid_dict = dict(regions_cluster.filter(['KOMNAVN', 'centroid']).values)
    if sl == 'omicron':
        centroid_dict['Aarhus'] = Point(centroid_dict['Aarhus'].x , centroid_dict['Aarhus'].y - 0.1)
        centroid_dict['Odense'] = Point(centroid_dict['Odense'].x + 0.1, centroid_dict['Odense'].y )
        centroid_dict['Holbæk'] = Point(centroid_dict['Holbæk'].x + 0.05  , centroid_dict['Holbæk'].y )
    if sl == 'Eta':
        centroid_dict['Odense'] = Point(centroid_dict['Odense'].x  , centroid_dict['Odense'].y )
    if sl == 'wildtype':
        centroid_dict['Roskilde'] = Point(centroid_dict['Roskilde'].x  , centroid_dict['Roskilde'].y + 0.1)
        centroid_dict['Holbæk'] = Point(centroid_dict['Holbæk'].x  , centroid_dict['Holbæk'].y + 0.05)

    cases_dict = dict(regions_cluster.filter(['KOMNAVN', 'len']).values)
    region_dict = dict(regions_cluster.filter(['KOMNAVN', 'REGIONNAVN']).values)
    cluster_start_nodes = [u for u, v in variant_crossovers]
    curve_x_list = []
    curve_y_list = []
    
    for kom in tqdm(regions_cluster_outbreak['KOMNAVN'].unique()):
        if kom not in cluster_start_nodes:
            continue
    
    
        kom_connections = [u[1] for u in variant_crossovers if u[0] == kom]
        if region_dict[kom] == 'Region Hovedstaden':
            if sl == 'omicron':
                start_x, start_y = (12.32226, 55.85625)
            elif sl == 'wildtype':
                start_x, start_y = (12.32226-0.1, 55.85625+0.1)
            elif sl == 'Delta':
                start_x, start_y = (12.32226, 55.85625)
        else:
            start_x, start_y = centroid_dict[kom].x, centroid_dict[kom].y
        for conn in kom_connections:
            
            if conn == 'Høje-Taastrup':
                conn = 'Høje Taastrup'
           
            if  np.isnan(cases_dict[kom] + cases_dict[conn]):
                continue
            if (conn, kom) in crossover_counts_dict.keys():
                if ((crossover_counts_dict[(kom, conn)] + crossover_counts_dict[(conn, kom)]) <= 5) and region_dict[conn] != 'Region Hovedstaden':
                    continue
            elif (crossover_counts_dict[(kom, conn)] <= 5) and region_dict[conn] != 'Region Hovedstaden':
                continue
            if region_dict[conn] == 'Region Hovedstaden':
                if sl == 'omicron':
                    end_x, end_y = (12.32226, 55.85625)
                elif sl == 'wildtype':
                    end_x, end_y = (12.32226-0.1, 55.85625+0.1)
                elif sl == 'Delta':
                    end_x, end_y = (12.32226+0.1, 55.85625-0.1)
            else:
                end_x, end_y = centroid_dict[conn].x, centroid_dict[conn].y
            curve_x, curve_y = draw_curve(np.array((start_x, start_y)), np.array((end_x, end_y)))
            curve_x_list += [curve_x]
            curve_y_list += [curve_y]
            # print(curve_x)
    curve_dict_variant[sl]['curve_x_list'] = curve_x_list
    curve_dict_variant[sl]['curve_y_list'] = curve_y_list
    regions_cluster_outbreak_variant_dict[sl] = regions_cluster_outbreak
    regions_cluster_variant_dict[sl] = regions_cluster

In [ ]:
largest_cluster_dict[sl].keys()
type_colour_dict = {'cluster_type' : ['school', 'workplace', 'both', 'small_cluster'] , 'colour' : ['blue', 'cyan', 'darkorange', 'forestgreen']}

largest_cluster_df_capital_dict = {}
def get_cluster_type_kommune(kommune, df):
    df_kommune = df.filter(pl.col('kommune_capital') == kommune)
    df_kommune_school = df_kommune.filter(~pl.col('school').is_null())
    df_kommune_workplace = df_kommune.filter(~pl.col('workplace').is_null())
    max_school_outbreak = df_kommune_school.group_by('school').len().max().select('len').item()
    if max_school_outbreak == None:
        max_school_outbreak = 0
    max_workplace_outbreak = df_kommune_workplace.group_by('workplace').len().max().select('len').item()
    if max_workplace_outbreak == None:
        max_workplace_outbreak = 0
    if max_workplace_outbreak >=20 and max_school_outbreak >= 20:
        cluster_type = 'both'
    elif max_school_outbreak > max_workplace_outbreak and max_school_outbreak >= 6:
        cluster_type = 'school'
    elif max_school_outbreak < max_workplace_outbreak and max_workplace_outbreak >= 6:
        cluster_type = 'workplace'
    elif max_school_outbreak < 6 and max_workplace_outbreak < 6:
        cluster_type = 'small_cluster'
    return cluster_type, max_school_outbreak, max_workplace_outbreak, len(df_kommune)



for sl in sl_list:
    largest_cluster_df_copy = largest_cluster_dfs[sl].clone()
    capital_kommuner = list(largest_cluster_df_copy.filter(pl.col('region') == 'Hovedstaden').select('kommune').unique().to_numpy().flatten())
    largest_cluster_df_capital = largest_cluster_df_copy.with_columns(pl.when(pl.col('kommune').is_in(capital_kommuner))
                .then(pl.lit('Hovedstaden'))
                .otherwise(pl.col('kommune'))).rename({'literal' : 'kommune_capital'})
    
    
    largest_cluster_df_capital_dict[sl] = largest_cluster_df_capital
    kommune_capital_pandas = largest_cluster_df_capital_dict[sl].select('kommune', 'kommune_capital').to_pandas()
    regions_cluster_outbreak_variant_dict[sl] = regions_cluster_outbreak_variant_dict[sl].merge(kommune_capital_pandas, on = 'kommune', how = 'left')


In [ ]:
def background_colour(number):
    if ~np.isnan(number):
        return 'purple'
    else:
        return 'lightgrey'
regions_cluster
for sl in sl_list:
    regions_cluster = regions_cluster_variant_dict[sl]
combined_cluster_df = pl.concat([largest_cluster_dfs['wildtype'], largest_cluster_dfs['Alpha'] , 
          largest_cluster_dfs['Eta'], largest_cluster_dfs['Delta'], largest_cluster_dfs['omicron']]).group_by(pl.col('kommune')).len()
regions_cluster = regions.merge(combined_cluster_df.to_pandas(),
                                left_on = 'KOMNAVN',
                                right_on = 'kommune', 
                                how = 'left')
cluster_size = df.select(pl.col('len').sum()).item()
regions_cluster = regions_cluster.loc[regions_cluster['KOMNAVN'] != 'Bornholm']
regions_cluster.loc[regions_cluster['len']<=5, 'len']= np.nan





In [ ]:


variant_colour_dict = {'Delta' : 'mediumvioletred', 'wildtype' : 'saddlebrown', 'Alpha' : 'gold', 
                      'omicron' : 'darkcyan', 'Eta' : 'darkorange'}
cluster_type_colour_dict = dict(zip(*type_colour_dict.values()))

In [ ]:
def scale_colour_flat(n):
    if np.isnan(n):
        return np.nan
    elif n<5:
        return np.nan
    else:
        return 30

regions_cluster['flat_colour'] = regions_cluster['len'].apply(scale_colour_flat)
regions_cluster['flat_colour'] = regions_cluster['flat_colour'].replace({0 : np.nan})
regions_cluster['len'] = regions_cluster['len'].replace({0 : np.nan})

In [ ]:
color_palette = ['saddlebrown', 'gold', 'darkorange', 'mediumvioletred', 'darkcyan']

def generate_cluster_plot(plot_df, plot_dict, ax, xlims = [0, 10], 
                          log_scale = False, title = None, 
                          legend = False, xlabel = None, ylabel = 'Variant', 
                         dict_func = np.mean):
    
    means_dict = plot_df.mean().to_dict()
    sns.stripplot(plot_df, orient = 'h', ax = ax, alpha = 0.15, palette = color_palette)
    
    ax.scatter(means_dict['Wildtype'], 0,
            color = variant_colour_dict['wildtype'], s = marker_size, marker = marker_mean, zorder = 10,
                 label = 'Randomised \n Networks (Mean)', edgecolors = 'black')
    ax.scatter(means_dict['Alpha'], 1,
                color = variant_colour_dict['Alpha'], s = marker_size, marker = marker_mean, zorder = 10,
                      edgecolors = 'black')
    ax.scatter(means_dict['Eta'], 2,
                color = variant_colour_dict['Eta'], s = marker_size, marker = marker_mean, zorder = 10,
                      edgecolors = 'black')
    ax.scatter(means_dict['Delta'], 3,
                color = variant_colour_dict['Delta'], s = marker_size, marker = marker_mean, zorder = 10,
                      edgecolors = 'black')
    ax.scatter(means_dict['Omicron'], 4,
                color = variant_colour_dict['omicron'], s = marker_size, marker = marker_mean, zorder = 10,
                      edgecolors = 'black')
    
    ax.scatter(dict_func(plot_dict['wildtype']), 0,
                color = variant_colour_dict['wildtype'], s = marker_size, label = 'Non-Randomised \n Network', marker = marker_true,  edgecolors = 'black')
    ax.scatter(dict_func(plot_dict['Alpha']), 1,
                color = variant_colour_dict['Alpha'], s = marker_size, marker = marker_true,  edgecolors = 'black')
    ax.scatter(dict_func(plot_dict['Eta']), 2,
                color = variant_colour_dict['Eta'], s = marker_size, marker = marker_true,  edgecolors = 'black')
    ax.scatter(dict_func(plot_dict['Delta']), 3,
                color = variant_colour_dict['Delta'], s = marker_size, marker = marker_true,  edgecolors = 'black')
    ax.scatter(dict_func(plot_dict['omicron']), 4,
                color = variant_colour_dict['omicron'], s = marker_size, marker = marker_true, edgecolors = 'black')
    
    ax.plot(np.array((dict_func(plot_dict['wildtype']), means_dict['Wildtype'])),
                   np.array((0, 0)), color = variant_colour_dict['wildtype'], linewidth = 2, zorder = 0)
    ax.plot(np.array((dict_func(plot_dict['Alpha']), means_dict['Alpha'])),
                   np.array((1, 1)), color = variant_colour_dict['Alpha'], linewidth = 2, zorder = 0)
    ax.plot(np.array((dict_func(plot_dict['Eta']), means_dict['Eta'])), 
                   np.array((2, 2)), color = variant_colour_dict['Eta'], linewidth = 2, zorder = 0)
    ax.plot(np.array((dict_func(plot_dict['Delta']), means_dict['Delta'])), 
                   np.array((3, 3)), color = variant_colour_dict['Delta'], linewidth = 2, zorder = 0)
    ax.plot(np.array((dict_func(plot_dict['omicron']), means_dict['Omicron'])),
                   np.array((4, 4)), color = variant_colour_dict['omicron'], linewidth = 2, zorder = 0)


    if log_scale:
        ax.set_xscale('log')
    if legend:
        ax.legend(loc = 'center right', frameon = True, title = 'Key:', fontsize = 8)
    
    ax.grid(alpha = 0.5)
    ax.set_xlim(xlims)
    ax.set_xlabel(xlabel, fontsize = 12)
    ax.set_ylabel(ylabel, fontsize = 12)
    ax.set_title(title, fontsize = 14)
    ax.set_yticklabels(['B.1.177', 'Alpha', 'Eta', 'Delta', 'Omicron'])
    

In [ ]:

fig, axes = plt.subplot_mosaic([['A', 'A', 'B', 'C'], ['A', 'A', 'D', 'E']], figsize = (20, 12), layout = 'constrained')
kom_cluster_dict_variant = {}
region_outbreak_colors_dict = {}
marker_size = 100
marker_mean = 'o'
marker_true = 's'

for sl in sl_list:

    gplot = regions_cluster.plot(ax = axes['A'],
                         column = 'flat_colour',
                         cmap = cmap,
                         missing_kwds = dict(color = 'lightgrey'),
                         vmin = 5,
                         vmax = 100, 
                        legend = False
                        )
    curve_x_list = curve_dict_variant[sl]['curve_x_list']
    curve_y_list = curve_dict_variant[sl]['curve_y_list']
    
    for c in range(len(curve_x_list)):
        if sl == 'Delta':
            continue
        curve_x = curve_x_list[c]
        curve_y = curve_y_list[c]
        axes['A'].plot(curve_x, curve_y, color = variant_colour_dict[sl], linewidth = 1, alpha = 1)

    regions_cluster_outbreak = regions_cluster_outbreak_variant_dict[sl]
    largest_cluster_df_capital = largest_cluster_df_capital_dict[sl]
    if 'kommune_capital' in largest_cluster_df_capital.columns:
        kommune_col = 'kommune_capital'
    else:
        kommune_col = 'kommune'
    kom_cluster_dict =  {kommune_col : [], 'cluster_type' : []}
    for kom in largest_cluster_df_capital_dict[sl].select(kommune_col).unique().to_numpy().flatten():
        kom_cluster_dict[kommune_col] += [kom]
        kom_cluster_dict['cluster_type'] += [get_cluster_type_kommune(kom, largest_cluster_df_capital)[0]]
    kom_cluster_df = pd.DataFrame(kom_cluster_dict)  
    type_colour_df = pd.DataFrame(type_colour_dict)
    kom_cluster_df = kom_cluster_df.merge(type_colour_df, on = 'cluster_type', how = 'left')
    kom_cluster_dict_variant[sl] = kom_cluster_df
    if kommune_col == 'kommune_capital':
        regions_cluster_outbreak_capital = regions_cluster_outbreak.dissolve(by = kommune_col)
    else:
        regions_cluster_outbreak_capital = regions_cluster_outbreak
        
    regions_cluster_outbreak_capital['centroid'] = regions_cluster_outbreak_capital.geometry.centroid
    regions_cluster_outbreak_colours = regions_cluster_outbreak_capital.merge(kom_cluster_df, on = kommune_col, how = 'left')
    if kommune_col == 'kommune_capital':
        regions_cluster_outbreak_colours.loc[regions_cluster_outbreak_colours[kommune_col] == 
            'Hovedstaden']['len'] = regions_cluster_outbreak.loc[regions_cluster_outbreak[kommune_col] == 
                                        'Hovedstaden'].drop_duplicates().len.sum() # Sum Hovedstaden - this is a bit hacky!
    outbreak_sizes = regions_cluster_outbreak_colours['len'].values * 3
    outbreak_sizes[outbreak_sizes > 400] = 400
    region_outbreak_colors_dict[sl] = regions_cluster_outbreak_colours
    for c, val in enumerate(regions_cluster_outbreak_colours[['kommune_capital', 'centroid']].values):
        centroid = val[1]
        kom = val[0]
        if sl == 'omicron':
            
            if kom == 'Aarhus':
                axes['A'].scatter(centroid.x , centroid.y - 0.1, 
                                     color = regions_cluster_outbreak_colours['colour'].values[c],
                                     s = outbreak_sizes[c],
                                     zorder = 10, 
                                     edgecolors = 'black') 
            elif kom == 'Odense':
                axes['A'].scatter(centroid.x +0.1, centroid.y, 
                                     color = regions_cluster_outbreak_colours['colour'].values[c],
                                     s = outbreak_sizes[c],
                                     zorder = 10, 
                                     edgecolors = 'black') 
            elif kom == 'Holbæk':
                axes['A'].scatter(centroid.x+ 0.05, centroid.y , 
                                     color = regions_cluster_outbreak_colours['colour'].values[c],
                                     s = outbreak_sizes[c],
                                     zorder = 10, 
                                     edgecolors = 'black') 
            else:
                axes['A'].scatter(centroid.x, centroid.y, 
                                  color = regions_cluster_outbreak_colours['colour'].values[c],
                                  s = outbreak_sizes[c],
                                  zorder = 10,
                                  edgecolors = 'black')
                
                
        elif sl == 'wildtype' and kom == 'Hovedstaden':
            axes['A'].scatter(12.32226 - 0.1, 55.85625 + 0.1, 
                             color = regions_cluster_outbreak_colours['colour'].values[c],
                             s = outbreak_sizes[c],
                             zorder = 10, 
                             edgecolors = 'black') 
        elif sl == 'wildtype' and kom == 'Roskilde':
            axes['A'].scatter(centroid.x, centroid.y + 0.1, 
                             color = regions_cluster_outbreak_colours['colour'].values[c],
                             s = outbreak_sizes[c],
                             zorder = 10, 
                             edgecolors = 'black') 
        elif sl == 'wildtype' and kom == 'Holbæk':
            axes['A'].scatter(centroid.x, centroid.y + 0.05, 
                             color = regions_cluster_outbreak_colours['colour'].values[c],
                             s = outbreak_sizes[c],
                             zorder = 10, 
                             edgecolors = 'black') 
        elif sl == 'Delta' and kom == 'Hovedstaden':
            axes['A'].scatter(12.32226 + 0.1, 55.85625 - 0.1, 
                             color = regions_cluster_outbreak_colours['colour'].values[c],
                             s = outbreak_sizes[c],
                             zorder = 10, 
                             edgecolors = 'black') 

            circle_delta = Point(12.32226 + 0.1, 55.85625 - 0.1).buffer(.15)
            circle_delta = shapely.affinity.scale(circle_delta, 1, .55)
            circle_delta_gp = gp.GeoSeries(circle_delta, crs = regions_cluster.crs )
            circle_delta_gp.crs = regions_cluster.crs

            
        elif sl == 'Eta':
            axes['A'].scatter(centroid.x-0.025, centroid.y , 
                             color = regions_cluster_outbreak_colours['colour'].values[c],
                             s = outbreak_sizes[c],
                             zorder = 10, 
                             edgecolors = 'black') 
            circle_eta = Point(centroid.x-0.025, centroid.y).buffer(.115)
            circle_eta = shapely.affinity.scale(circle_eta, 1, .55)
            circle_eta_gp = gp.GeoSeries(circle_eta, crs = regions_cluster.crs )
            circle_eta_gp.crs = regions_cluster.crs
        
        else:
            axes['A'].scatter(centroid.x, centroid.y, color = regions_cluster_outbreak_colours['colour'].values[c],
                                                          s = outbreak_sizes[c], zorder = 10, edgecolors = 'black')


regions_cluster_outbreak_colours.loc[regions_cluster_outbreak_colours[kommune_col] == 'Hovedstaden'].plot(ax = axes['A'],
                                                                                                                facecolor = 'none', 
                                                                                                                edgecolor = 'black', 
                                                                                                               linewidth = 2, 
                                                                                                               label = 'Capital Region of Denmark')



regions.loc[regions['REGIONNAVN']!= 'Region Hovedstaden'].dissolve('KOMNAVN').plot(ax = axes['A'],
                                      facecolor = 'none', 
                                      edgecolor = 'black',
                                      linewidth = 0.5)

# Create legend and add dummy plots for labelling

for ct in cluster_type_colour_dict.keys():
    
    axes['A'].scatter(0, 0, color = cluster_type_colour_dict[ct], s = 40, 
                      zorder = 10, edgecolors = 'black', label = ct.replace('_', ' ').capitalize())
axes['A'].plot(np.array((0, 0.1)), np.array((0, 0)), color = 'black', linewidth = 2, label = 'Capital Region of Denmark')


for i, txt in enumerate(city_labels):
    if txt == 'Copenhagen':
        axes['A'].scatter(city_points_x[i], city_points_y[i], color = 'red', s = 20, zorder = 20)
        axes['A'].annotate('Copenhagen', (city_points_x[i] + 0.05, city_points_y[i] + 0.05), 
                           bbox = dict(facecolor = 'white', edgecolor = 'black'), zorder = 20 )
    elif txt == 'Aarhus':
        axes['A'].scatter(city_points_x[i], city_points_y[i], color = 'red', s = 20, zorder = 20)
        axes['A'].annotate(txt, (city_points_x[i] + 0.05, city_points_y[i] - 0.075), 
                           bbox = dict(facecolor = 'white', edgecolor = 'black'), zorder = 20 )
    else:
        axes['A'].annotate(txt, (city_points_x[i] + 0.05, city_points_y[i] + 0.05), 
                           bbox = dict(facecolor = 'white', edgecolor = 'black'), zorder = 20 )
        axes['A'].scatter(city_points_x[i], city_points_y[i], color = 'red', s = 20, zorder = 20)

for sl in sl_list:
    if sl == 'Eta': 
        circle_eta_gp.to_crs(regions_cluster.crs).boundary.plot(ax = axes['A'], color = variant_colour_dict[sl], label = sl.capitalize())
    elif sl == 'Delta':
        circle_delta_gp.boundary.plot(ax = axes['A'], color = variant_colour_dict[sl], label = sl.capitalize())
    elif sl == 'wildtype':
        axes['A'].plot(np.array((0, 0.1)), np.array((0, 0)), color = variant_colour_dict[sl], linewidth = 1, alpha = 1, label = 'B.1.177')
    else:
        axes['A'].plot(np.array((0, 0.1)), np.array((0, 0)), color = variant_colour_dict[sl], linewidth = 1, alpha = 1, label = sl.capitalize())


handles, labels = axes['A'].get_legend_handles_labels()
unique = dict(zip(labels, handles))
axes['A'].get_xaxis().set_visible(False)
axes['A'].get_yaxis().set_visible(False)
axes['A'].legend(unique.values(), unique.keys(), title = 'Key:', frameon = True)
axes['A'].set_ylim([54.5, 58])
axes['A'].set_xlim([8, 13])
axes['A'].set_title('Largest Settings Clusters \n by Variant', fontsize = 14)


#### That's panel A done! 
# Proportion of individuals in a cluster - Panel B

cluster_sizes_sum_df = pd.DataFrame.from_dict(cluster_sizes_sum_dict)
cluster_sizes_sum_df.columns = cluster_sizes_sum_df.columns.str.capitalize()
cluster_proportions_dict = dict([(key, cluster_sizes_real_dict[key] / lineage_sizes_dict[key] ) for key in cluster_sizes_len_dict_real.keys()])
plot_title_B = 'Proportion of Individuals \n Belonging to a Cluster'
plot_xlabel_B = 'Proportion'
generate_cluster_plot(cluster_sizes_sum_df, cluster_proportions_dict, axes['B'], xlims = None,
                      title = plot_title_B, xlabel = plot_xlabel_B, log_scale = False, dict_func = np.sum)
# Number of clusters

cluster_sizes_len_df = pd.DataFrame.from_dict(cluster_sizes_len_dict)
cluster_sizes_len_df.columns = cluster_sizes_len_df.columns.str.capitalize()
plot_title_C = 'Number of Clusters'
plot_xlabel_C = 'Number of Clusters'
generate_cluster_plot(cluster_sizes_len_df, cluster_sizes_len_dict_real, axes['C'], xlims = None,
                      title = plot_title_C, xlabel = plot_xlabel_C, log_scale = True)


# Maximum cluster size - panel E

cluster_sizes_max_df = pd.DataFrame.from_dict(cluster_sizes_max_dict)
cluster_sizes_max_df.columns = cluster_sizes_max_df.columns.str.capitalize()
plot_title_D = 'Largest Cluster Size'
plot_xlabel_D = 'Number of Individuals'
generate_cluster_plot(cluster_sizes_max_df, cluster_sizes_real_dict, axes['D'], xlims = [1, 2500],
                      title = plot_title_D, xlabel = plot_xlabel_D, log_scale = True, dict_func = np.max)


# Kommuner 
cluster_sizes_kommuner_df = pd.DataFrame.from_dict(cluster_kommuner_dict_variant)
cluster_sizes_kommuner_df.columns = cluster_sizes_kommuner_df.columns.str.capitalize()
# cluster_sizes_kommuner_df = cluster_sizes_kommuner_df.rename({'Wildtype' : 'B.1.177'})
plot_title_E = 'Mean number of Municipalities \n Spanned by a Cluster'
plot_xlabel_E = 'Number of Municipalities'
generate_cluster_plot(cluster_sizes_kommuner_df, cluster_kommuner_dict_real, axes['E'], xlims = [0, 8],
                      title = plot_title_E, xlabel = plot_xlabel_E, legend = True)

for label, ax in axes.items():
    ax.set_title(label, loc = 'left', fontsize = 14)
plt.tight_layout()
plt.savefig('../../Figures/clusters_figure_new_map.pdf')